# Linking order data with DXF geometry

This notebook links the customer orders (`customers_sales.csv`) with the geometry values of the DXF files (folder `10991360`) and exports the result to Excel.

## Overview

| Step | Block | Delivers |
|:-:|---|---|
| 1 | Evaluate DXF files | `df` (one row per DXF) |
| 2 | Read order data | `orders` (one row per order) |
| 2a | Check names for spelling errors | `orders` (names unified), `name_fixes`, `name_review_df` |
| 3 | Validate order data | Console report (changes nothing) |
| 4 | Validate DXF values | Console report and `invalid_geo` |
| 5 | Clean with log | `orders_clean`, `df_clean`, `removed_report`, `dup_dxf`, `suspicious` |
| 6 | Merge | `merged` (one row per order incl. DXF values) |
| 7 | Check and export | Excel file with 5 sheets |


In [12]:
import re
from difflib import SequenceMatcher
import pandas as pd
import os
from helpers import *
import openpyxl

## 1. Evaluate DXF files

**What happens:** All `.dxf` files in the folder `10991360` are opened one after another. Four metrics are computed for each file and collected as one row.

**Builds on:**
- `helpers.py` (`get_dxf_bounds`, `get_cutting_length`, `get_number_of_holes`)
- the folder `10991360` containing the DXF files

**Delivers:** `df` with one row per DXF and the columns

| Column | Meaning |
|---|---|
| `file` | File name, e.g. `Geo (10).DXF` (later join key) |
| `bounding_box_width` / `_height` | Width and height of the enclosing box |
| `cutting_length` | Total cutting length |
| `num_holes` | Number of holes (inner contours) |

**Notes:**
- Files not ending in `.dxf` (case-insensitive) are skipped.
- If a calculation fails, `helpers.py` returns `None`. In `df` this becomes `NaN` and shows up in step 4.
- The Excel export of `df` is commented out; `df` is only used in memory.

In [13]:

path = "10991360_editet"

rows = []

for file in os.listdir(path):
    file_path = os.path.join(path, file)

    # Optional: process only DXF files
    if not file.lower().endswith(".dxf"):
        continue

    bounding_box_width, bounding_box_height = get_dxf_bounds(file_path)
    cutting_length = get_cutting_length(file_path)
    num_holes = get_number_of_holes(file_path)
    bounding_box_area = bounding_box_width * bounding_box_height
    rectangularity = get_rectangularity_ratio(file_path)
    num_pierce_points = get_pierce_points(file_path)

    rows.append({
        "file": file,
        "bounding_box_width": bounding_box_width,
        "bounding_box_height": bounding_box_height,
        "bounding_box_area": bounding_box_area,
        "cutting_length": cutting_length,
        "num_holes": num_holes,
        "rectangularity": rectangularity,
        "num_pierce_points": num_pierce_points
    })

df = pd.DataFrame(rows)
# df.to_excel("ergebnisse.xlsx", index=False, engine="openpyxl")

## 2. Read order data

**What happens:** The CSV is read and the columns are converted to clean data types.
- `sep=None, engine="python"` detects the delimiter (tab, `;` or `,`) automatically.
- Column names and text columns are stripped of leading/trailing spaces, so the later comparison does not fail because of invisible whitespace.
- `order_quantity` is converted to a number. Anything unreadable (e.g. `"1.600"` or `"n/a"`) becomes `NaN` (`errors="coerce"`).

**Builds on:** `customers_sales.csv` with the columns `customer_name`, `customer_branch`, `file_name`, `order_quantity`

**Delivers:** `orders` with one row per order. Nothing is deleted; the row count equals the CSV.

In [14]:
# Bestelldaten einlesen
orders = pd.read_csv("customers_sales.csv", sep=None, engine="python")
orders.columns = orders.columns.str.strip()

for col in ["customer_name", "customer_branch", "file_name"]:
    orders[col] = orders[col].astype("string").str.strip()
orders["order_quantity"] = pd.to_numeric(orders["order_quantity"], errors="coerce")

# print(orders.shape)
# orders.head()

## 2a. Detect and unify spelling errors in customer and branch names

**What happens:** Similarly spelled names (e.g. `AutoPart AG` and `AutoParts AG`) are detected and mapped to the **most frequent spelling**. This happens **before** validation, so duplicate check and merge work with unified names.

- **Similarity:** `difflib.SequenceMatcher` (0 to 1) on a simplified spelling (lowercase, without punctuation and extra spaces).
- **From `SIM_THRESHOLD` (0.90)** names are merged automatically. **Between `REVIEW_THRESHOLD` (0.80) and 0.90** they are only reported, nothing is changed.
- **Branches** are compared **only within the same customer** (`Frankfurt Branch` at two different customers is allowed).
- **Safeguards against wrong merges:**
  - Different **numbers** (`Branch 1` / `Branch 2`) are never merged.
  - A different **legal form** (`MetalForm AG` / `MetalForm SA`) is never merged automatically, only reported for review, since these may be different companies.
- **Own overrides:** In `MANUAL_MAP` you can define mappings, e.g. `{"AutoPart AG": "AutoParts AG"}`. With `{"X": "X"}` you prevent `X` from ever being merged.
- **`APPLY = False`:** report only, `orders` stays unchanged. Recommendation: run it this way first, check the report, then set it to `True`.

**Builds on:** `orders` from step 2

**Delivers:**
- `orders` with corrected names (if `APPLY = True`). The original spellings are kept in `customer_name_original` and `customer_branch_original`.
- `name_fixes`: all corrections (original, correction, number of rows, similarity)
- `name_review_df`: similar names that were **not** merged and should be reviewed

**Notes:**
- The mapping is a heuristic. Very short names or very similar real company names can be grouped wrongly. Look at the correction list before continuing.
- After unification, steps 3 and 5 may detect **more exact duplicates** than before, because rows that were spelled differently now match. The original spelling is shown next to them in `removed_report` and `suspicious`.
- Typos in `file_name` are not checked. Orders without a matching DXF are shown in step 6.

In [15]:
# Schreibfehler in Kunden-/Filialnamen erkennen und vereinheitlichen
# ---------- Einstellungen ----------
SIM_THRESHOLD = 0.90      # ab dieser Ähnlichkeit wird automatisch zugeordnet
REVIEW_THRESHOLD = 0.80   # ab dieser Ähnlichkeit (aber unter SIM_THRESHOLD) nur melden
APPLY = True              # False = nur berichten, orders bleibt unverändert
MANUAL_MAP = {}           # Ausnahmen/Vorgaben, z. B. {"AutoPart AG": "AutoParts AG"}
                          # oder {"MetalForm SA": "MetalForm SA"} = nie zusammenführen


# Rechtsformen: Unterschiedliche Rechtsform (AG vs. SA) = vermutlich andere Firma, nie automatisch zusammenführen
LEGAL_FORMS = {"ag", "sa", "gmbh"}


def _norm(s):
    """Klein, ohne Satzzeichen/Mehrfach-Leerzeichen, damit nur echte Tippfehler zählen."""
    return re.sub(r"[^\w]+", " ", str(s).lower()).strip()


def _similar(a, b):
    na, nb = _norm(a), _norm(b)
    # Unterschiedliche Zahlen (z. B. "Filiale 1" vs. "Filiale 2") sind nie derselbe Wert
    if re.findall(r"\d+", na) != re.findall(r"\d+", nb):
        return 0.0
    ratio = SequenceMatcher(None, na, nb).ratio()
    if {w for w in na.split() if w in LEGAL_FORMS} != {w for w in nb.split() if w in LEGAL_FORMS}:
        return min(ratio, SIM_THRESHOLD - 0.001)   # höchstens "bitte prüfen", nie automatisch
    return ratio


def build_mapping(values):
    """Ordnet seltene Schreibweisen der häufigsten ähnlichen Schreibweise zu."""
    counts = values.dropna().value_counts()
    order = sorted(counts.index, key=lambda v: (-counts[v], v))   # häufigste zuerst
    canon, mapping, review = [], {}, []
    for v in order:
        if v in MANUAL_MAP:
            target = MANUAL_MAP[v]
            if target not in canon:
                canon.append(target)
            mapping[v] = target
            continue
        best, best_sim = None, 0.0
        for c in canon:
            s = _similar(v, c)
            if s > best_sim:
                best, best_sim = c, s
        if best is not None and best_sim >= SIM_THRESHOLD:
            mapping[v] = best
        else:
            canon.append(v)
            mapping[v] = v
            if best is not None and best_sim >= REVIEW_THRESHOLD:
                review.append((v, best, round(best_sim, 3)))
    return mapping, counts, review


fix_rows, review_rows = [], []

# ---------- 1) customer_name ----------
orders["customer_name_original"] = orders["customer_name"]
name_map, name_counts, name_review = build_mapping(orders["customer_name"])
for old, new in name_map.items():
    if old != new:
        fix_rows.append({"feld": "customer_name", "kunde": new, "original": old, "korrigiert": new,
                         "anzahl_zeilen": int(name_counts[old]),
                         "aehnlichkeit": round(_similar(old, new), 3)})
for a, b, s in name_review:
    review_rows.append({"feld": "customer_name", "kunde": "-", "wert_a": a, "wert_b": b, "aehnlichkeit": s})
customer_final = orders["customer_name"].map(name_map).astype("string")

# ---------- 2) customer_branch (nur innerhalb desselben Kunden vergleichen) ----------
orders["customer_branch_original"] = orders["customer_branch"]
branch_final = orders["customer_branch"].copy()
for cust, grp in orders.groupby(customer_final, dropna=True):
    b_map, b_counts, b_review = build_mapping(grp["customer_branch"])
    branch_final.loc[grp.index] = grp["customer_branch"].map(b_map)
    for old, new in b_map.items():
        if old != new:
            fix_rows.append({"feld": "customer_branch", "kunde": cust, "original": old, "korrigiert": new,
                             "anzahl_zeilen": int(b_counts[old]),
                             "aehnlichkeit": round(_similar(old, new), 3)})
    for a, b, s in b_review:
        review_rows.append({"feld": "customer_branch", "kunde": cust, "wert_a": a, "wert_b": b, "aehnlichkeit": s})

name_fixes = pd.DataFrame(fix_rows, columns=["feld", "kunde", "original", "korrigiert", "anzahl_zeilen", "aehnlichkeit"])
name_review_df = pd.DataFrame(review_rows, columns=["feld", "kunde", "wert_a", "wert_b", "aehnlichkeit"])

# ---------- 3) Anwenden + Bericht ----------
if APPLY:
    orders["customer_name"] = customer_final
    orders["customer_branch"] = branch_final.astype("string")

print(f"Korrekturen {'ANGEWENDET' if APPLY else 'nur vorgeschlagen (APPLY=False)'}: {len(name_fixes)} Schreibweisen "
      f"({int(name_fixes['anzahl_zeilen'].sum()) if len(name_fixes) else 0} Zeilen betroffen)")
if len(name_fixes):
    print(name_fixes.to_string(index=False))
print(f"\nÄhnlich, aber NICHT zusammengeführt (bitte prüfen): {len(name_review_df)}")
if len(name_review_df):
    print(name_review_df.to_string(index=False))

Korrekturen ANGEWENDET: 2 Schreibweisen (4 Zeilen betroffen)
         feld        kunde     original   korrigiert  anzahl_zeilen  aehnlichkeit
customer_name AutoParts AG  AutoPart AG AutoParts AG              3         0.957
customer_name MetalForm SA Metalform SA MetalForm SA              1         1.000

Ähnlich, aber NICHT zusammengeführt (bitte prüfen): 0


## 3. Validate order data (check only)

**What happens:** The orders are checked for irregularities and the count per problem is printed. **Nothing is changed or deleted.**

| Check | Meaning |
|---|---|
| Missing values | empty fields per column |
| Invalid quantities | `NaN`, ≤ 0 or with decimals |
| Exact duplicates | all 4 columns identical (`keep=False` marks **all** involved rows, including the original) |
| Same order, different quantity | same customer, branch and file, but different quantity. This can be a repeat order and is therefore only reported |

**Builds on:** `orders` from step 2a

**Delivers:** Console output. Also creates `cols` (the 4 check columns, reused in step 5) as well as `invalid_qty`, `exact_dups` and `same_order_diff_qty` for inspecting details.

In [16]:
# Bestelldaten validieren
cols = ["customer_name", "customer_branch", "file_name", "order_quantity"]

print("Fehlende Werte:\n", orders[cols].isna().sum(), "\n")

invalid_qty = orders[
    orders["order_quantity"].isna()
    | (orders["order_quantity"] <= 0)
    | (orders["order_quantity"] % 1 != 0)
]
print("Ungültige Mengen:", len(invalid_qty))

exact_dups = orders[orders.duplicated(subset=cols, keep=False)]
print("Exakte Duplikate:", len(exact_dups))

# gleiche Kunde/Filiale/Datei, aber andere Menge (nur zur Prüfung, wird nicht gelöscht)
key = ["customer_name", "customer_branch", "file_name"]
same_order_diff_qty = orders[
    orders.duplicated(subset=key, keep=False) & ~orders.duplicated(subset=cols, keep=False)
]
print("Gleiche Order, andere Menge:", len(same_order_diff_qty))

Fehlende Werte:
 customer_name      0
customer_branch    0
file_name          0
order_quantity     0
dtype: int64 

Ungültige Mengen: 0
Exakte Duplikate: 8
Gleiche Order, andere Menge: 0


## 4. Validate DXF values (check only)

**What happens:** The computed DXF values are checked for plausibility: file names unique? Missing values? Width, height or cutting length ≤ 0?

The file name in `df` is stripped of whitespace (as for the orders in step 2), so the join in step 6 works reliably.

**Builds on:** `df` from step 1

**Delivers:**
- Console report
- `geo_cols` (the three geometry columns, reused in steps 5 and 6)
- `invalid_geo`: the affected rows, displayed as a table below the cell

**Note:** `num_holes = 0` is valid (part without holes). Therefore `num_holes` is only checked for missing values, not for ≤ 0.

In [17]:
# DXF-Werte validieren
geo_cols = ["bounding_box_width", "bounding_box_height", "cutting_length"]

df["file"] = df["file"].astype("string").str.strip()

print("DXF-Dateien:", len(df))
print("Doppelte Dateinamen:", df["file"].duplicated().sum())
print("Fehlende Werte:\n", df[geo_cols + ["num_holes"]].isna().sum(), "\n")

invalid_geo = df[df[geo_cols].isna().any(axis=1) | (df[geo_cols] <= 0).any(axis=1)]
print("Fehlerhafte Geometriewerte:", len(invalid_geo))
invalid_geo

DXF-Dateien: 80
Doppelte Dateinamen: 0
Fehlende Werte:
 bounding_box_width     0
bounding_box_height    0
cutting_length         0
num_holes              0
dtype: int64 

Fehlerhafte Geometriewerte: 0


,file,bounding_box_width,bounding_box_height,bounding_box_area,cutting_length,num_holes,rectangularity,num_pierce_points


## 5. Clean with log

**What happens:** The actual cleaning happens here; every removal is recorded.

1. **Remove exact duplicates:** A row is deleted **only** if customer, branch, file **and** quantity are identical. The first row is kept. `gleich_wie_index` shows which row the duplicate belongs to.
2. **Flag suspicious rows, do not delete:** Missing values, quantities ≤ 0 or with decimals are kept and end up in `suspicious`. The raw text of the quantity from the CSV is shown next to it (`rohwert_menge`) to see what was really there.
3. **Clean DXF values:** Duplicate file names are removed (first row is kept). Invalid geometry values (`NaN` or ≤ 0) are set to `NaN` and listed; the row stays.

**Builds on:** `orders` (step 2a), `cols` (step 3), `df` and `geo_cols` (step 4); additionally the CSV, which is read again here as raw text (`dtype=str`)

**Delivers:**

| Variable | Content |
|---|---|
| `orders_clean` | Orders without exact duplicates (basis for the merge) |
| `removed_report` | removed orders with reason and `gleich_wie_index` |
| `suspicious` | kept but suspicious orders |
| `df_clean` | DXF values without duplicate file names, invalid values as `NaN` |
| `dup_dxf` | removed duplicate DXF rows |

**Notes:**
- `raw` and `orders` must have the same index. This holds as long as no rows are filtered in step 2.
- `order_quantity` is converted to integers only if there are no suspicious rows. Otherwise the type is kept, because `NaN` cannot be an `int`.
- Two rows with identically empty fields count as duplicates.

In [18]:
# Rohwerte der CSV (als Text), damit man bei nicht lesbaren Mengen sieht, was dort stand
raw = pd.read_csv("customers_sales.csv", sep=None, engine="python", dtype=str)
raw.columns = raw.columns.str.strip()

# --- 1) Gedroppt werden NUR exakte Duplikate (alle 4 Spalten gleich) ---
dup_mask = orders.duplicated(subset=cols, keep="first")

# Index der ersten Zeile, mit der das Duplikat übereinstimmt
first_idx = orders.assign(_idx=orders.index).groupby(cols, dropna=False)["_idx"].transform("min")

removed_report = orders[dup_mask].copy()
removed_report["grund"] = "Exaktes Duplikat (alle 4 Spalten gleich)"
removed_report["gleich_wie_index"] = first_idx[dup_mask]

print(f"Orders: {len(orders)} -> {len(orders) - len(removed_report)} "
      f"({len(removed_report)} entfernt)\n")
if len(removed_report):
    print("Entfernte Zeilen:")
    print(removed_report.to_string(), "\n")

orders_clean = orders[~dup_mask].copy()

# --- 2) Auffällige Zeilen: werden NICHT entfernt, nur angezeigt ---
suspicious_mask = (
    orders_clean[cols].isna().any(axis=1)
    | (orders_clean["order_quantity"] <= 0)
    | (orders_clean["order_quantity"] % 1 != 0)
)
suspicious = orders_clean[suspicious_mask].copy()
suspicious["rohwert_menge"] = raw.loc[suspicious.index, "order_quantity"]

print(f"Auffällige Zeilen (BEHALTEN, bitte prüfen): {len(suspicious)}")
if len(suspicious):
    print(suspicious.to_string())

# Menge nur in int umwandeln, wenn alle Werte gültig sind
if not suspicious_mask.any():
    orders_clean["order_quantity"] = orders_clean["order_quantity"].astype(int)

# --- 3) DXF-Werte: nur doppelte Dateinamen entfernen, ungültige Werte auf NaN setzen ---
dup_dxf = df[df.duplicated(subset="file", keep="first")]
print(f"\nEntfernte doppelte DXF-Zeilen: {len(dup_dxf)}")
if len(dup_dxf):
    print(dup_dxf.to_string())

df_clean = df.drop_duplicates(subset="file").copy()
for c in geo_cols:
    bad = ~(df_clean[c] > 0)
    if bad.any():
        print(f"\nUngültige Werte in {c} -> NaN gesetzt:")
        print(df_clean.loc[bad, ["file", c]].to_string())
    df_clean.loc[bad, c] = pd.NA

Orders: 79 -> 75 (4 entfernt)

Entfernte Zeilen:
        customer_name   customer_branch     file_name  order_quantity customer_name_original customer_branch_original                                     grund  gleich_wie_index
7   PrecisionCut GmbH  Stuttgart Branch  Geo (15).DXF            4000      PrecisionCut GmbH         Stuttgart Branch  Exaktes Duplikat (alle 4 Spalten gleich)                 3
49       AutoParts AG     Munich Branch  Geo (55).DXF            9250           AutoParts AG            Munich Branch  Exaktes Duplikat (alle 4 Spalten gleich)                42
62  PrecisionCut GmbH  Stuttgart Branch  Geo (56).DXF            4800      PrecisionCut GmbH         Stuttgart Branch  Exaktes Duplikat (alle 4 Spalten gleich)                50
68       AutoParts AG     Berlin Branch  Geo (12).DXF            8800           AutoParts AG            Berlin Branch  Exaktes Duplikat (alle 4 Spalten gleich)                 4 

Auffällige Zeilen (BEHALTEN, bitte prüfen): 0

Entfernte do

## 6. Merge

**What happens:** The DXF values are attached to the orders. `file_name` (order) is joined with `file` (DXF).

- `how="left"`: **Every order is kept**, even without a matching DXF (then `NaN` in the DXF columns).
- `validate="many_to_one"`: Several orders may use the same DXF, but each DXF may appear only once in `df_clean`. Otherwise pandas raises an error instead of silently multiplying rows.
- `indicator=True`: helper column `_merge` showing whether a row was found (`both`) or not (`left_only`). It is removed again after the evaluation.
- The duplicate key column `file` is dropped.

**Builds on:** `orders_clean` and `df_clean` from step 5, `geo_cols` from step 4

**Delivers:** `merged` with one row per order and the order columns plus `bounding_box_width`, `bounding_box_height`, `cutting_length`, `num_holes`. The console output shows how many orders remained without a DXF and how many rows have missing DXF values.

**Note:** If a file name does not match exactly (typo or `.dxf` instead of `.DXF`), the row stays without DXF values. The comparison is case-sensitive.

In [19]:
# Zusammenführen
merged = orders_clean.merge(
    df_clean,
    left_on="file_name",
    right_on="file",
    how="left",
    validate="many_to_one",   # mehrere Orders pro DXF erlaubt, jede DXF nur einmal
    indicator=True,
).drop(columns="file")

print(merged["_merge"].value_counts())
print("Orders ohne DXF:", (merged["_merge"] == "left_only").sum())
print("Zeilen mit fehlenden DXF-Werten:", merged[geo_cols].isna().any(axis=1).sum())

merged = merged.drop(columns="_merge")
merged.head()

_merge
both          75
left_only      0
right_only     0
Name: count, dtype: int64
Orders ohne DXF: 0
Zeilen mit fehlenden DXF-Werten: 0


,customer_name,customer_branch,file_name,order_quantity,customer_name_original,customer_branch_original,bounding_box_width,bounding_box_height,bounding_box_area,cutting_length,num_holes,rectangularity,num_pierce_points
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600,MetalForm SA,Frankfurt Branch,1600.219048,305.140135,488291.056038,3438.024977,0,0.082199,1
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650,AutoParts AG,Munich Branch,853.830000,333.401400,284668.117362,3217.207505,5,0.724271,6
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800,PrecisionCut GmbH,Stuttgart Branch,164.000000,145.120000,23799.680000,1742.517642,7,0.600032,8
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000,PrecisionCut GmbH,Stuttgart Branch,193.800891,138.442589,26830.297074,1302.774283,2,0.685885,3
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800,AutoParts AG,Berlin Branch,937.150697,74.006665,69355.398059,2764.783512,11,0.865878,12


## 7. Check and export to Excel

**What happens:**
1. **Info:** Counts DXF files for which no order exists (display only).
2. **Safety check:** The `assert` aborts if the merge changed the row count. It compares with `orders_clean`, so losses from step 5 are only visible in the log there.
3. **Log tables** for the Excel file are prepared (`original_index` is the row index in the DataFrame, starting at 0; the CSV row is roughly index + 2).
4. **Export** to `customers_sales_with_dxf_data.xlsx` with five sheets; afterwards the column widths are adjusted automatically (maximum 60 characters).

**Builds on:** `merged` (step 6), `orders_clean` and `df_clean` (step 5), `removed_report` and `dup_dxf` (step 5), `name_fixes` and `name_review_df` (step 2a)

**Delivers:** the Excel file `customers_sales_with_dxf_data.xlsx`

| Sheet | Content |
|---|---|
| `Daten` | `merged`, the finished main table |
| `Entfernte Orders` | dropped duplicates with reason |
| `Entfernte DXF` | duplicate DXF rows with reason |
| `Namenskorrekturen` | all unified spellings from step 2a |
| `Namen zu prüfen` | similar but not merged names from step 2a |

**Notes:**
- Sheets with 0 removed rows are kept (header only). This way you can see that the check was done.
- The file is overwritten on every run. If it is open in Excel, writing fails.
- The cell ends with `merged`, so the table is also displayed in the notebook.

In [20]:
# DXF-Dateien ohne Bestellung (nur zur Info)
unused = set(df_clean["file"]) - set(orders_clean["file_name"])
print("DXF ohne Bestellung:", len(unused))

assert len(merged) == len(orders_clean), "Merge hat Zeilen verändert!"

# ---------- Tabellen der entfernten Zeilen vorbereiten ----------
orders_drop = removed_report.reset_index().rename(columns={"index": "original_index"})

dxf_drop = dup_dxf.reset_index().rename(columns={"index": "original_index"})
dxf_drop["grund"] = "Doppelter Dateiname (erste Zeile behalten)"

# ---------- Export ----------
with pd.ExcelWriter("customers_sales_with_dxf_data.xlsx", engine="openpyxl") as writer:
    merged.to_excel(writer, sheet_name="Daten", index=False)
    orders_drop.to_excel(writer, sheet_name="Entfernte Orders", index=False)
    dxf_drop.to_excel(writer, sheet_name="Entfernte DXF", index=False)
    name_fixes.to_excel(writer, sheet_name="Namenskorrekturen", index=False)
    name_review_df.to_excel(writer, sheet_name="Namen zu prüfen", index=False)

    # Spaltenbreiten automatisch anpassen
    for ws in writer.sheets.values():
        for column_cells in ws.columns:
            width = max(len(str(c.value)) if c.value is not None else 0 for c in column_cells)
            ws.column_dimensions[column_cells[0].column_letter].width = min(width + 2, 60)

print(f"Export fertig: customers_sales_with_dxf_data.xlsx"
      f"({len(merged)} Orders, {len(orders_drop)} entfernte Orders, {len(dxf_drop)} entfernte DXF-Zeilen, "
      f"{len(name_fixes)} Namenskorrekturen)")
merged

DXF ohne Bestellung: 5
Export fertig: customers_sales_with_dxf_data.xlsx(75 Orders, 4 entfernte Orders, 0 entfernte DXF-Zeilen, 2 Namenskorrekturen)


,customer_name,customer_branch,file_name,order_quantity,customer_name_original,customer_branch_original,bounding_box_width,bounding_box_height,bounding_box_area,cutting_length,num_holes,rectangularity,num_pierce_points
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600,MetalForm SA,Frankfurt Branch,1600.219048,305.140135,4.882911e+05,3438.024977,0,0.082199,1
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650,AutoParts AG,Munich Branch,853.830000,333.401400,2.846681e+05,3217.207505,5,0.724271,6
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800,PrecisionCut GmbH,Stuttgart Branch,164.000000,145.120000,2.379968e+04,1742.517642,7,0.600032,8
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000,PrecisionCut GmbH,Stuttgart Branch,193.800891,138.442589,2.683030e+04,1302.774283,2,0.685885,3
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800,AutoParts AG,Berlin Branch,937.150697,74.006665,6.935540e+04,2764.783512,11,0.865878,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,AutoParts AG,Munich Branch,Geo (78).DXF,8150,AutoParts AG,Munich Branch,386.361049,290.000000,1.120447e+05,5581.874650,52,0.636032,53
71,PrecisionCut GmbH,Stuttgart Branch,Geo (79).DXF,3750,PrecisionCut GmbH,Stuttgart Branch,36.553638,36.553638,1.336168e+03,147.032435,4,0.770113,5
72,TechFab Industries,Hamburg Branch,Geo (8).DXF,12050,TechFab Industries,Hamburg Branch,801.705747,344.180571,2.759315e+05,6111.748725,15,0.429933,16
73,MetalForm SA,Frankfurt Branch,Geo (80).DXF,2100,MetalForm SA,Frankfurt Branch,1968.901094,832.434782,1.638982e+06,16821.059941,15,0.611551,16
